# 04 &mdash; Prove it is correct, then break it on purpose

> **Reading order.** These notebooks are a course, not a reference. Each one
> assumes the previous. `01` builds the tensor vocabulary, `02` builds the layers,
> `03` trains them on Cora, `04` proves they are correct and breaks them
> deliberately, `05` moves to graph-level tasks.
>
> Everything imports the local `gnn_lab` package -- edit the package and re-run
> the cell. The self-contained Kaggle run lives in `kaggle/`.

**Goal.** Two things. First, establish that the hand-written layers are actually
right -- not "they train", but "they compute the same function as a reference
implementation". Second, run the ablations that turn design choices from
assertions into measurements.

This is the notebook that makes the rest of the repo trustworthy.

---

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))     # so `import gnn_lab` works from notebooks/

import torch
import torch.nn as nn
import torch.nn.functional as F

from gnn_lab import (GCNScratch, GraphSAGEScratch, GATScratch, GINScratch,
                     GCNConvScratch, SAGEConvScratch, GATConvScratch, GINConvScratch,
                     load_cora, load_mutag, triangle_square_dataset, karate_club,
                     toy_graph, path_graph, two_communities,
                     train_node, train_node_multi_seed, train_graph_classification,
                     evaluate_node, set_seed, get_device, count_parameters,
                     accuracy, macro_f1, majority_baseline,
                     mean_pairwise_cosine, dirichlet_energy)
from gnn_lab.ops import (scatter_add, scatter_mean, scatter_max, scatter_softmax,
                         gcn_norm, add_self_loops, remove_self_loops, to_undirected,
                         degree, dense_to_edge_index)
from gnn_lab.layers.pooling import collate_graphs, global_pool, graph_batches
from gnn_lab import equivalence, viz

DEVICE = get_device()
RESULTS = os.path.join("..", "results")
os.makedirs(RESULTS, exist_ok=True)

print("torch  :", torch.__version__)
print("device :", DEVICE)
print("PyG    :", "available" if equivalence.PYG_AVAILABLE else f"missing ({equivalence.PYG_IMPORT_ERROR})")
set_seed(0)


## 1. Why a correctness proof is necessary

Writing a GNN from scratch is easy. Writing one that is *right* is not, because
**the failure mode is silent**.

Swap `src` and `dst` in a message pass and:

* the shapes stay valid,
* the model still trains,
* the loss still decreases,
* the accuracy still looks plausible,
* and the answer is quietly wrong.

Nothing throws. There is no exception to catch. This actually happened in the
first draft of `GCNConvScratch` in this repo, and it was caught by a test, not
by a runtime error.

So the check has to be external: build the same layer in PyTorch Geometric, copy
the weights across verbatim, and assert the outputs agree.

$$\max \big|\, f_{\text{mine}}(x, A) - f_{\text{PyG}}(x, A) \,\big| < \epsilon$$

### Why weight copying works at all

Because `gnn_lab` deliberately names its parameters the way PyG does:

| layer | shared parameter names |
|---|---|
| GCN | `lin`, `bias` |
| GraphSAGE | `lin_l`, `lin_r` |
| GAT | `lin`, `att_src`, `att_dst`, `bias` |
| GIN | `nn`, `eps` |

That is not an aesthetic choice. It turns the check into
`pyg.load_state_dict(mine.state_dict())` instead of a brittle hand-written key
mapping that has to be updated every time PyG ships a release.

### Why float64

In float32 a `1e-5` tolerance would pass even for a subtly wrong
implementation -- rounding noise alone is that large. In float64, agreement to
`~1e-12` is evidence the two are computing the *same expression*, not merely
similar ones.

In [ ]:
import inspect
print(inspect.getsource(equivalence.check_gcn))

## 2. Run everything

In [ ]:
results = equivalence.run_all(root="../data", include_data=True, verbose=True)
print()
print("VERDICT:", "all checks passed" if equivalence.all_passed(results)
      else "SOMETHING FAILED -- see the table above")

In [ ]:
print(f"{'status':<7}{'max|diff|':>12}{'tolerance':>12}  check")
print("-" * 96)
for r in results:
    d = "   -" if r.max_diff is None else f"{r.max_diff:.3e}"
    t = "   -" if r.tolerance is None else f"{r.tolerance:.0e}"
    print(f"{r.status:<7}{d:>12}{t:>12}  {r.name}")

print()
from collections import Counter
for status, n in sorted(Counter(r.status for r in results).items()):
    print(f"  {status:<6} {n}")

### What each family of checks is proving

* **`scatter_*` vs python loop** -- the primitives match their mathematical
  definition. No library involved, so this always runs. It is the foundation
  everything else stands on.
* **`gcn_norm` coefficients** -- the `D^-1/2 (A+I) D^-1/2` weights are identical
  edge by edge.
* **`GCNConvScratch` / `SAGEConvScratch` / `GATConvScratch` / `GINConvScratch`**
  -- each layer reproduces PyG bit-for-bit given the same weights.
* **`global_pool`** -- batching readouts match PyG's `global_{add,mean,max}_pool`.
* **`load_cora` vs `Planetoid`** -- the loader reproduces PyG's tensors exactly:
  features, labels, edge ordering and all three masks.

### If PyG is not installed

Every PyG check reports `SKIP` with a reason rather than crashing. The
from-scratch code has **no** PyG dependency; it is used only as a reference
oracle here. To install:

```bash
pip install torch_geometric
```

## 3. End-to-end parity

Layer-level equality is necessary but not sufficient -- a model could still
assemble the layers wrongly. This closes the loop: same data, same architecture,
same seed, same hyper-parameters, both implementations trained end to end.

The two accuracies will not be *bitwise* identical (dropout and the optimiser
make the trajectories diverge chaotically) but they should land within a couple
of points. A systematic gap would mean one of the two is computing something
different.

In [ ]:
if equivalence.PYG_AVAILABLE:
    from torch_geometric.nn import GCNConv

    class PyGGCN(nn.Module):
        def __init__(self, f, h, c):
            super().__init__()
            self.c1, self.c2 = GCNConv(f, h), GCNConv(h, c)
        def forward(self, x, edge_index):
            x = F.dropout(x, p=0.5, training=self.training)
            x = F.relu(self.c1(x, edge_index))
            x = F.dropout(x, p=0.5, training=self.training)
            return self.c2(x, edge_index)

    cora = load_cora(source="planetoid", row_normalize=False, verbose=False).to(DEVICE)

    accs = {}
    for label, factory in (("scratch", lambda: GCNScratch(cora.num_features, 16, cora.num_classes,
                                                          num_layers=2, dropout=0.5)),
                           ("pyg",     lambda: PyGGCN(cora.num_features, 16, cora.num_classes))):
        runs = []
        for seed in (0, 1, 2):
            set_seed(seed)
            r = train_node(factory(), cora, model_name=label, epochs=100, device=DEVICE, seed=seed)
            runs.append(r.test_acc_at_best_val)
        accs[label] = (sum(runs)/len(runs), runs)
        print(f"  {label:8} test acc {accs[label][0]*100:.2f}%  seeds {[f'{v*100:.1f}' for v in runs]}")

    gap = abs(accs["scratch"][0] - accs["pyg"][0])
    print(f"\n  gap: {gap*100:.2f} points -> {'consistent' if gap < 0.03 else 'INVESTIGATE'}")
else:
    print("PyG not installed -- skipping the end-to-end parity check.")

---
## 4. Over-smoothing: why GNNs are shallow

The obvious question about any message-passing model is "why only 2 layers?".
CNNs go 50 or 100 deep. The answer is that message passing is **diffusion**, and
diffusion has a fixed point: given enough steps, every node converges to the
same value.

Two quantitative measures make this concrete rather than hand-wavy:

**Mean pairwise cosine similarity** between node embeddings. Rises toward 1.0 as
layers stack. At 1.0 every node has the same representation and no classifier
can separate them.

**Graph Dirichlet energy** $\sum_{(i,j) \in E} \|h_i - h_j\|^2$, normalised per
edge. Diffusion monotonically *decreases* it -- neighbouring nodes are pulled
together every layer. As it approaches 0, features stop carrying class
information.

These two move in opposite directions, which is the signature of collapse.

In [ ]:
cora = load_cora(source="planetoid", row_normalize=False, verbose=False).to(DEVICE)

depth_rows = []
for depth in (1, 2, 3, 4, 6, 8, 12, 16):
    accs, coss, ens = [], [], []
    for seed in (0, 1):
        set_seed(seed)
        m = GCNScratch(cora.num_features, 16, cora.num_classes, num_layers=depth, dropout=0.5)
        r = train_node(m, cora, model_name=f"GCN-{depth}L", epochs=200, device=DEVICE, seed=seed)
        accs.append(r.test_acc_at_best_val)
        m.eval()
        with torch.no_grad():
            h = m.embeddings(cora.x, cora.edge_index)
        coss.append(mean_pairwise_cosine(h))
        ens.append(dirichlet_energy(h, cora.edge_index, cora.num_nodes))
    row = {"depth": depth, "variant": "plain",
           "test_acc": sum(accs)/len(accs),
           "cosine": sum(coss)/len(coss),
           "energy": sum(ens)/len(ens)}
    depth_rows.append(row)
    print(f"  {depth:2d} layers   test {row['test_acc']*100:5.2f}%   "
          f"cosine {row['cosine']:.4f}   energy {row['energy']:.3e}")

viz.plot_oversmoothing([r["depth"] for r in depth_rows], depth_rows,
                       path=f"{RESULTS}/04_oversmoothing.png")

### Watching it happen, layer by layer

Same trained network, one forward pass, capturing the hidden state after each
layer. No retraining involved -- this is purely the effect of stacking.

In [ ]:
set_seed(0)
deep = GCNScratch(cora.num_features, 16, cora.num_classes, num_layers=8,
                  dropout=0.0).to(DEVICE)
train_node(deep, cora, model_name="GCN-8L", epochs=200, device=DEVICE)

deep.eval()
hs = [cora.x]
with torch.no_grad():
    h = cora.x
    for i, conv in enumerate(deep.convs):
        h = conv(h, cora.edge_index)
        if i < len(deep.convs) - 1:
            h = F.relu(h)
        hs.append(h)

print(f"{'layer':>6}{'shape':>14}{'cosine':>10}{'energy':>14}")
for k, hh in enumerate(hs):
    print(f"{k:>6}{str(tuple(hh.shape)):>14}{mean_pairwise_cosine(hh):>10.4f}"
          f"{dirichlet_energy(hh, cora.edge_index, cora.num_nodes):>14.4e}")

viz.plot_layerwise_similarity(hs, cora.edge_index,
                              path=f"{RESULTS}/04_layerwise.png")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(19, 4.4))
for ax, k in zip(axes, (0, 1, 2, 8)):
    if k < len(hs):
        viz.plot_tsne(hs[k].cpu(), cora.y.cpu(),
                      title=("input features" if k == 0 else f"after {k} layers"),
                      ax=ax, seed=0)
fig.suptitle("Over-smoothing: class clusters merge as depth grows", y=1.04)
fig.tight_layout()
viz.save_or_show(fig, f"{RESULTS}/04_tsne_depth.png", show=False)

### Two standard fixes

Neither is a cure; both shift the collapse later, which is enough to make the
mechanism obvious.

**Residual connections** -- even if layer `k` has diffused everything into mush,
the skip path still carries the shallower representation to the classifier.

**Jumping Knowledge (JK-Net)** -- concatenate *every* layer's output and classify
from the concatenation, so the model chooses the receptive field instead of
committing to the last layer's.

In [ ]:
class ResidualGCN(nn.Module):
    """h <- h + dropout(relu(conv(h))).  `depth` = number of conv layers, as for the others.

    An earlier version built depth-1 convs (so "depth 1" was a graph-free MLP) and put a
    LayerNorm after every add, which let the raw-feature path drown out the graph signal
    and held it at 50-58% at every depth. Input dropout matters too: with 140 labels the
    input projection overfits without it.
    """
    def __init__(self, f, h, c, depth, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.input_proj = nn.Linear(f, h)
        self.convs = nn.ModuleList([GCNConvScratch(h, h) for _ in range(depth)])
        self.out = nn.Linear(h, c)

    def forward(self, x, edge_index):
        h = self.input_proj(F.dropout(x, p=self.dropout, training=self.training))
        for conv in self.convs:
            h = h + F.dropout(F.relu(conv(h, edge_index)),
                              p=self.dropout, training=self.training)
        return self.out(h)

    def embeddings(self, x, edge_index):
        h = self.input_proj(x)
        for conv in self.convs:
            h = h + F.relu(conv(h, edge_index))
        return h


class JumpingKnowledgeGCN(nn.Module):
    def __init__(self, f, h, c, depth, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        dims = [f] + [h] * depth
        self.convs = nn.ModuleList([GCNConvScratch(dims[i], dims[i+1]) for i in range(depth)])
        self.out = nn.Linear(h * depth, c)

    def forward(self, x, edge_index):
        hs = []
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
            hs.append(x)
        return self.out(torch.cat(hs, dim=-1))

    def embeddings(self, x, edge_index):
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
        return x


variants = {"plain": lambda d: GCNScratch(cora.num_features, 16, cora.num_classes,
                                           num_layers=d, dropout=0.5),
            "residual": lambda d: ResidualGCN(cora.num_features, 16, cora.num_classes, d),
            "jumping_knowledge": lambda d: JumpingKnowledgeGCN(cora.num_features, 16,
                                                               cora.num_classes, d)}

all_rows = list(depth_rows)
for vname, factory in variants.items():
    if vname == "plain":
        continue
    print(f"=== {vname} ===")
    for depth in (2, 4, 8, 16):
        set_seed(0)
        m = factory(depth).to(DEVICE)
        r = train_node(m, cora, model_name=f"{vname}-{depth}L", epochs=200, device=DEVICE)
        m.eval()
        with torch.no_grad():
            h = m.embeddings(cora.x, cora.edge_index)
        row = {"depth": depth, "variant": vname, "test_acc": r.test_acc_at_best_val,
               "cosine": mean_pairwise_cosine(h),
               "energy": dirichlet_energy(h, cora.edge_index, cora.num_nodes)}
        all_rows.append(row)
        print(f"  {depth:2d} layers   test {row['test_acc']*100:5.2f}%   "
              f"cosine {row['cosine']:.4f}   energy {row['energy']:.3e}")

viz.plot_oversmoothing([r["depth"] for r in all_rows], all_rows,
                       path=f"{RESULTS}/04_depth_variants.png")

---
## 5. The other ablations

### 5a. Aggregator

Same GraphSAGE architecture, three `combine` operators. This is the
message-passing abstraction reduced to its single degree of freedom.

In [ ]:
print(f"{'aggregator':<12}{'test acc':>12}{'macro F1':>12}")
print("-" * 36)
for aggr in ("mean", "max", "sum"):
    accs, f1s = [], []
    for seed in (0, 1, 2):
        set_seed(seed)
        m = GraphSAGEScratch(cora.num_features, 16, cora.num_classes,
                             num_layers=2, dropout=0.5, aggr=aggr)
        r = train_node(m, cora.to(DEVICE), model_name=f"SAGE-{aggr}", epochs=200,
                       device=DEVICE, seed=seed)
        accs.append(r.test_acc_at_best_val); f1s.append(r.test_f1_at_best_val)
    print(f"{aggr:<12}{sum(accs)/len(accs)*100:>11.2f}%{sum(f1s)/len(f1s)*100:>11.2f}%")
print()
print("`sum` is degree-sensitive, so hubs dominate -- usually visible as higher")
print("variance across seeds as well as a different mean.")

### 5b. Self-loops

Both GCN and GAT depend on a node being able to see itself. Remove that and find
out how much of the accuracy was the node's own features all along.

In [ ]:
for loops in (True, False):
    accs = []
    for seed in (0, 1, 2):
        set_seed(seed)
        m = GCNScratch(cora.num_features, 16, cora.num_classes, num_layers=2,
                       dropout=0.5, add_self_loops=loops)
        r = train_node(m, cora.to(DEVICE), model_name="GCN", epochs=200,
                       device=DEVICE, seed=seed)
        accs.append(r.test_acc_at_best_val)
    print(f"  add_self_loops={loops!s:5}  test {sum(accs)/len(accs)*100:5.2f}%")
print()
print("Without self-loops a node can only be reconstructed from its neighbours,")
print("which on a sparse citation graph is a substantial handicap.")

### 5c. Those 302 duplicate edges

A data-hygiene question with a measurable answer, because most tutorials pick
one behaviour without mentioning it. PyG de-duplicates; `tkipf/gcn` does not.

In [ ]:
for dedupe in (True, False):
    d = load_cora(source="planetoid", row_normalize=False, dedupe=dedupe,
                  verbose=False).to(DEVICE)
    accs = []
    for seed in (0, 1, 2):
        set_seed(seed)
        m = GCNScratch(d.num_features, 16, d.num_classes, num_layers=2, dropout=0.5)
        r = train_node(m, d, model_name="GCN", epochs=200, device=DEVICE, seed=seed)
        accs.append(r.test_acc_at_best_val)
    print(f"  dedupe={dedupe!s:5}  edges {d.num_edges:5d}  test {sum(accs)/len(accs)*100:5.2f}%")
print()
print("The effect is small -- but it is real, and it is the kind of discrepancy")
print("that makes two papers' 'same' experiment disagree by half a point.")

---
## 6. What is GAT's attention actually doing?

The claim is that GAT learns to weight informative neighbours more heavily.
That is falsifiable: split the learned attention coefficients into edges joining
two nodes of the **same** class and edges joining **different** classes, and
compare the distributions.

If attention is doing its job, the same-class distribution sits to the right.

**Expect this to be a messy result.** On homophilous Cora the separation is
visible but far from clean. That is the honest finding, and a useful corrective
to figures that make attention look like a perfect edge classifier.

In [ ]:
set_seed(0)
gat = GATScratch(cora.num_features, 8, cora.num_classes, heads=8, dropout=0.0).to(DEVICE)
train_node(gat, cora, model_name="GAT", epochs=200, device=DEVICE)

ei_att, att = gat.attention_weights(cora.x, cora.edge_index, layer=0)
att = att.detach().cpu()
src_i, dst_i = ei_att[0].cpu(), ei_att[1].cpu()
y_cpu = cora.y.cpu()
same = y_cpu[src_i] == y_cpu[dst_i]

per_edge = att.mean(dim=1)     # average across the 8 heads
print(f"edges carrying attention: {per_edge.numel()}")
print(f"  same-class  edges: {int(same.sum()):5d}   mean attention {float(per_edge[same].mean()):.4f}")
print(f"  cross-class edges: {int((~same).sum()):5d}   mean attention {float(per_edge[~same].mean()):.4f}")
print(f"  ratio same/cross : {float(per_edge[same].mean() / per_edge[~same].mean()):.3f}")

viz.plot_attention_distribution(per_edge[same], per_edge[~same],
                                path=f"{RESULTS}/04_attention.png")

In [ ]:
print("attention per head, same-class vs cross-class edges:")
print(f"{'head':>5}{'same':>10}{'cross':>10}{'ratio':>9}")
for hnd in range(att.size(1)):
    s = float(att[same, hnd].mean()); d = float(att[~same, hnd].mean())
    print(f"{hnd:>5}{s:>10.4f}{d:>10.4f}{s/max(d,1e-12):>9.3f}")
print()
print("If the ratios differ across heads, the heads have specialised -- which is")
print("the stated purpose of multi-head attention. If they are all near-identical,")
print("the heads are redundant and you are paying 8x the parameters for nothing.")

sub = torch.arange(60)
mask = torch.isin(src_i, sub) & torch.isin(dst_i, sub)
viz.draw_graph(ei_att[:, mask], labels=y_cpu[sub],
               highlight_edges=per_edge[mask].float(),
               title="GAT attention on a Cora subgraph (edge width = attention)",
               path=f"{RESULTS}/04_attention_graph.png", node_size=110)

### Is the attention entropy informative?

A node whose attention is nearly uniform is not really "attending" to anything.
Measure the entropy per node and see how many nodes have a peaked distribution.

In [ ]:
ent = -(att * (att + 1e-12).log()).sum(dim=0)          # [N, heads]
deg = scatter_add(torch.ones(ei_att.size(1)), ei_att[1].cpu(), cora.num_nodes)
max_ent = deg.clamp(min=2).log().unsqueeze(-1)          # entropy of a uniform distribution
norm_ent = (ent / max_ent).mean(dim=1)

print(f"normalised attention entropy per node: mean {float(norm_ent.mean()):.4f}")
print(f"  (1.0 = perfectly uniform, i.e. no attention at all; 0.0 = one-hot)")
print(f"  nodes below 0.9 : {int((norm_ent < 0.9).sum())} / {cora.num_nodes}")
print(f"  nodes below 0.7 : {int((norm_ent < 0.7).sum())} / {cora.num_nodes}")
print()
print("Most nodes sit close to uniform, which is consistent with the overlapping")
print("distributions above: attention helps, but it is a soft nudge rather than")
print("a hard selection of relevant edges.")

## 7. Summary

* **Correctness is checkable.** Every layer matches `torch_geometric` in float64
  with weights copied verbatim, and the loader reproduces PyG's tensors exactly.
  That is the only way to catch the silent `src`/`dst` class of bug.
* **Over-smoothing is measurable.** Cosine similarity up, Dirichlet energy down,
  accuracy collapsing -- three curves telling one story. Residual and JK
  connections delay it; they do not remove it.
* **Design choices have sizes.** The aggregator, the self-loop, and even 302
  duplicate edges in the raw data all move the number. Now you know by how much.
* **Attention is soft.** Same-class edges do get more weight than cross-class
  ones, but the distributions overlap heavily and most nodes stay near uniform.

**Next:** `05_graph_level_gin.ipynb` -- a different task shape, where graphs are
the examples rather than the container, and the expressiveness theorem becomes
an experiment.